# V14 patient-tag audit — private, CPU only
Check all training studies before new specialist training. No model fitting, pixel decoding, report identity reconstruction, or leaderboard submission.
PatientID consistency is not proof the anonymizer preserved patient identity between examinations. Raw patient IDs are never exported.


In [ ]:
import types
patient_audit = types.ModuleType("patient_audit")
exec('"""Read-only patient-tag audit. No fitting, pixel decoding, or identity reconstruction.\n\nOne first and one last DICOM header per series are checked; this is not an audit\nof every slice. PatientID-backed grouping is conditional on the anonymizer having\npreserved identity across examinations. That semantic guarantee is not inferred.\n"""\nimport hashlib\nimport json\nimport os\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport pandas as pd\nimport pydicom\n\nUID = \'StudyInstanceUID\'\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\nPLACEHOLDERS = {\'\', \'anonymous\', \'anonymized\', \'anon\', \'unknown\', \'none\', \'null\', \'0\', \'1\'}\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef inspect_study(uid, series_ids, images_root, salt):\n    tokens, errors, checked = set(), [], 0\n    for sid in sorted(series_ids):\n        files = sorted((images_root / uid / sid).glob(\'*.dcm\'))\n        if not files:\n            errors.append(\'series_has_no_dicom\')\n            continue\n        for file in sorted(set([files[0], files[-1]])):\n            try:\n                ds = pydicom.dcmread(file, stop_before_pixels=True,\n                                    specific_tags=[\'PatientID\', UID, \'SeriesInstanceUID\'])\n                checked += 1\n                pid = str(ds.get(\'PatientID\', \'\')).strip()\n                if pid.lower() in PLACEHOLDERS:\n                    errors.append(\'missing_or_placeholder_patient_id\')\n                elif pid == uid or pid == sid:\n                    errors.append(\'patient_id_is_study_or_series_id\')\n                else:\n                    # Equal IDs merge conservatively even across institutions.\n                    # Raw patient identifiers and the random salt are never exported.\n                    tokens.add(hashlib.sha256(salt + pid.encode()).hexdigest())\n                if str(ds.get(UID, \'\')) != uid or str(ds.get(\'SeriesInstanceUID\', \'\')) != sid:\n                    errors.append(\'dicom_path_uid_mismatch\')\n            except Exception as exc:\n                errors.append(\'header_read_error:\' + type(exc).__name__)\n    if len(tokens) != 1:\n        errors.append(\'inconsistent_or_absent_patient_id_within_study\')\n    return {\'StudyInstanceUID\': uid, \'GroupID\': next(iter(tokens)) if len(tokens) == 1 else \'\',\n            \'headers_checked\': checked, \'series_checked\': len(series_ids),\n            \'issues\': sorted(set(errors))}\n\n\ndef audit(root, output, workers=12):\n    root, output = Path(root), Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    train = pd.read_csv(root / \'train.csv\', dtype={UID: str})\n    series = pd.read_csv(root / \'train_series.csv\', dtype={UID: str, \'SeriesInstanceUID\': str})\n    if train[UID].isna().any() or train[UID].duplicated().any():\n        raise ValueError(\'Invalid study IDs\')\n    if set(train[UID]) != set(series[UID]) or series[\'SeriesInstanceUID\'].duplicated().any():\n        raise ValueError(\'Training/series coverage or uniqueness mismatch\')\n    grouped = series.groupby(UID)[\'SeriesInstanceUID\'].agg(list).to_dict()\n    salt, records = os.urandom(32), []\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        futures = {pool.submit(inspect_study, uid, grouped[uid], root / \'train_series\', salt): uid\n                   for uid in train[UID]}\n        for future in as_completed(futures):\n            records.append(future.result())\n            if len(records) % 250 == 0 or len(records) == len(train):\n                print(f\'PATIENT AUDIT {len(records)}/{len(train)} studies\', flush=True)\n    records.sort(key=lambda r: r[UID])\n    valid = [r for r in records if not r[\'issues\']]\n    mapping = pd.DataFrame([{UID: r[UID], \'GroupID\': r[\'GroupID\']} for r in valid], columns=[UID, \'GroupID\'])\n    mapping.to_csv(output / \'patient_tag_groups.csv\', index=False)\n    group_sizes = mapping.groupby(\'GroupID\').size()\n    counts = {}\n    for record in records:\n        for issue in record[\'issues\']:\n            counts[issue] = counts.get(issue, 0) + 1\n    gold = train[train[TARGETS].notna().all(axis=1)]\n    gold_map = gold[[UID] + TARGETS].merge(mapping, on=UID, how=\'left\', validate=\'one_to_one\')\n    eligible = len(valid) == len(train) and len(group_sizes) >= 20\n    report = {\n        \'status\': \'PATIENT_TAG_COVERAGE_COMPLETE_SEMANTICS_UNVERIFIED\' if eligible else \'PATIENT_GROUPING_FAILED\',\n        \'scope\': \'First and last header in every series; no pixel or report identity reconstruction.\',\n        \'studies\': len(train), \'series\': len(series), \'headers_checked\': sum(r[\'headers_checked\'] for r in records),\n        \'studies_with_usable_consistent_tags\': len(valid), \'patient_tag_groups\': len(group_sizes),\n        \'groups_with_multiple_studies\': int((group_sizes > 1).sum()),\n        \'largest_group_studies\': int(group_sizes.max()) if len(group_sizes) else 0,\n        \'issue_study_counts\': counts, \'gold_studies\': len(gold),\n        \'gold_patient_tag_groups\': int(gold_map[\'GroupID\'].nunique()),\n        \'missing_gold_group_ids\': int(gold_map[\'GroupID\'].isna().sum()),\n        \'input_sha256\': {name: digest(root / name) for name in [\'train.csv\', \'train_series.csv\']},\n        \'patient_map_sha256\': digest(output / \'patient_tag_groups.csv\'),\n        \'tag_coverage_gate_passed\': eligible,\n        \'patient_identity_preservation_independently_verified\': False,\n        \'baseline_v13_exclusion_verified\': False,\n        \'training_performed\': False, \'plus_0_02_verified\': False,\n        \'caveat\': \'Repeated examinations may have different anonymized PatientIDs. Header consistency alone cannot prove patient independence. Need host/source confirmation or an authoritative mapping.\',\n    }\n    (output / \'patient_metadata_audit.json\').write_text(json.dumps(report, indent=2) + \'\\n\', encoding=\'utf-8\')\n    (output / \'private_study_audit.json\').write_text(json.dumps(records) + \'\\n\', encoding=\'utf-8\')\n    print(json.dumps(report, indent=2), flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    import argparse\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--root\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    audit(args.root, args.output)\n', patient_audit.__dict__)
from pathlib import Path
candidates = [Path('/kaggle/input/rsna-knee-abnormality-detection'),
              Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')]
roots = [p for p in candidates if (p / 'train.csv').exists()]
if len(roots) != 1:
    raise RuntimeError('Expected exactly one competition root')
PATIENT_AUDIT = patient_audit.audit(roots[0], '/kaggle/working/patient_audit', workers=12)
print('NO TRAINING OR SUBMISSION. Patient identity semantics and V13 exclusions remain unverified.')
